# insurance-demand

**Conversion, retention, and price elasticity modelling for UK personal lines insurance.**

This notebook shows the core workflow in under two minutes: fit a conversion model on synthetic PCW quote data, read the implied price elasticity, build a demand curve, and find the profit-maximising price subject to an ENBP ceiling.

The problem: a perfectly calibrated risk model tells you the expected cost of a policy, but not whether the customer will accept the quoted price. That is what demand modelling adds. Without it, you cannot answer the most commercially important question in pricing: what price maximises expected profit rather than just matching the technical premium?

In [ ]:
!pip install -q insurance-demand polars numpy scikit-learn scipy statsmodels pandas

## Synthetic PCW quote data

The built-in generator produces realistic UK motor PCW quote data with a known true price elasticity (-2.0). It includes the key confounding structure: high-risk customers face higher technical premiums AND are less price-sensitive (fewer alternatives). This is why naive regression on absolute price produces a biased elasticity estimate.

In [ ]:
from insurance_demand.datasets import generate_conversion_data

df = generate_conversion_data(n_quotes=5_000, seed=42)

print(f"Quotes:         {len(df):,}")
print(f"Conversion rate: {df['converted'].mean():.1%}")
print(f"Avg quoted price: £{df['quoted_price'].mean():.0f}")
print(f"Avg tech premium: £{df['technical_premium'].mean():.0f}")
print()
df.head()

## Fit a conversion model

`ConversionModel` predicts P(buy | price, features). The price treatment is `log(quoted_price / technical_premium)` - the commercial loading rather than the absolute price. This makes the price coefficient comparable across risk segments and recovers the true elasticity within a few percent.

Including `rank_position_col` captures the discrete PCW rank effect: being cheapest versus second cheapest is worth more than the price gap alone would suggest.

In [ ]:
from insurance_demand import ConversionModel

model = ConversionModel(
    base_estimator="logistic",
    feature_cols=["age", "vehicle_group", "ncd_years", "area", "channel"],
    rank_position_col="rank_position",
)
model.fit(df)

probs = model.predict_proba(df)
print(f"Avg predicted conversion: {probs.mean():.1%}")
print(f"Avg observed conversion:  {df['converted'].mean():.1%}")
print()
print("Coefficient table (top 8 by magnitude):")
print(model.summary().head(8))

## Implied price elasticity

`price_elasticity()` returns the model-implied log-log elasticity: d log P(buy) / d log price. This is the naive elasticity from the fitted model - it is biased if there is residual confounding beyond what the price/technical_premium normalisation removes. For the debiased causal estimate, use `ElasticityEstimator` (requires the `dml` extra).

The true elasticity embedded in the DGP is -2.0. The correctly specified logistic model recovers it within about 4-5%.

In [ ]:
elasticity = model.price_elasticity(df)
print(f"Mean implied elasticity:   {elasticity.mean():.3f}")
print(f"Median implied elasticity: {elasticity.median():.3f}")
print(f"True DGP elasticity:       -2.0")
print()
print("One-way: observed vs fitted conversion by channel:")
print(model.oneway(df, "channel"))

## Build a demand curve

`DemandCurve` converts the elasticity estimate into a price-to-probability curve. We use the `semi_log` parametric form here (appropriate for binary outcomes). Supply the base price and base probability to anchor the curve, then evaluate it across a price range.

In [ ]:
import numpy as np
from insurance_demand import DemandCurve

# Use the mean implied elasticity and observed base point
base_price = float(df["quoted_price"].mean())
base_prob  = float(df["converted"].mean())
elas       = float(elasticity.mean())

curve = DemandCurve(
    elasticity=elas,
    base_price=base_price,
    base_prob=base_prob,
    functional_form="semi_log",
)

prices, p_buy = curve.evaluate(price_range=(base_price * 0.6, base_price * 1.5), n_points=50)

import polars as pl
demand_df = pl.DataFrame({"price": prices, "p_buy": p_buy})
print(f"At £{base_price:.0f} (base): P(buy) = {p_buy[25]:.3f}")
print()
print(demand_df.filter(pl.col("price") > base_price * 0.8).head(10))

## Find the profit-maximising price

`OptimalPrice` maximises expected profit per quote: `P(buy | price) x (price - expected_loss - expenses)`. The `enbp` parameter enforces the FCA PS21/11 ceiling for renewal pricing. For a new business segment there is no ENBP ceiling.

In [ ]:
from insurance_demand import OptimalPrice

# Assume expected loss = 65% of technical premium, 15% expense ratio
expected_loss = base_price * 0.65

opt = OptimalPrice(
    demand_curve=curve,
    expected_loss=expected_loss,
    expense_ratio=0.15,
    min_price=base_price * 0.70,
    max_price=base_price * 1.40,
)

result = opt.optimise()
print(f"Optimal price:             £{result.optimal_price:.2f}")
print(f"P(buy) at optimal:         {result.conversion_prob:.1%}")
print(f"Expected profit per quote: £{result.expected_profit:.2f}")
print(f"Margin per policy bound:   £{result.expected_margin:.2f}")
print(f"Active constraints:        {result.constraints_active or 'none'}")
print()
loading = result.optimal_price / base_price - 1
print(f"Price loading vs base:     {loading:+.1%}")

## What you should see

The optimal price sits above the technical premium. With elasticity around -2, a small increase above the technical rate still converts enough business that the margin gain outweighs the volume loss - up to a point. The loading depends on the elasticity: more elastic customers mean a smaller optimal loading.

This is single-segment pricing. For portfolio-level factor optimisation across many policies simultaneously, pass the demand curve callable (`curve.as_demand_callable()`) into `insurance-optimise` which solves the full constrained portfolio problem.

**GitHub:** https://github.com/burning-cost/insurance-demand  
**PyPI:** https://pypi.org/project/insurance-demand/